## Numerični izračun matrik pričakovanih časov srečanja za modele

In [1]:
import time
import networkx as nx
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import bicgstab

In [ ]:
def zgradi_multistohasticno_matriko(graf):
    """
    Iz multigrafa zgradi stohastično matriko P in pripravi preslikave vozlišč.
    """
    # Fiksiramo vrstni red vozlišč
    index_to_node = list(graf.nodes())
    node_to_index = {node: i for i, node in enumerate(index_to_node)}
    n = len(index_to_node)
    
    row_ind, col_ind, data = [], [], []
    
    for u in index_to_node:
        i = node_to_index[u]
        izhodne = list(graf.out_edges(u, data=True))
        
        if not izhodne:
            # Če vozlišče nima izhodov (absorbirajoče stanje)
            row_ind.append(i)
            col_ind.append(i)
            data.append(1.0)
            continue
            
        # Izračun stopenj r_e = 1 / t za vse paralelne povezave
        stopnje = [1.0 / podatki['weight'] for _, v, podatki in izhodne]
        skupna_stopnja = sum(stopnje)
        
        # Agregiramo verjetnosti med u in v za paralelne povezave
        verjetnosti_do_v = {}
        for (_, v, _), r in zip(izhodne, stopnje):
            j = node_to_index[v]
            verjetnosti_do_v[j] = verjetnosti_do_v.get(j, 0.0) + (r / skupna_stopnja)
            
        for j, p in verjetnosti_do_v.items():
            row_ind.append(i)
            col_ind.append(j)
            data.append(p)
            
    P = sp.csr_matrix((data, (row_ind, col_ind)), shape=(n, n))
    return P, index_to_node, node_to_index

def zgradi_stohasticno_matriko(graf):
    """
    Iz usmerjenega grafa (nx.DiGraph) zgradi stohastično matriko P in pripravi preslikave vozlišč.

    """
    # Fiksiramo vrstni red vozlišč
    index_to_node = list(graf.nodes())
    node_to_index = {node: i for i, node in enumerate(index_to_node)}
    n = len(index_to_node)

    row_ind, col_ind, data = [], [], []

    for u in index_to_node:
        i = node_to_index[u]
        izhodne = list(graf.out_edges(u, data=True))

        if not izhodne:
            # Če vozlišče nima izhodov (absorbirajoče stanje)
            row_ind.append(i)
            col_ind.append(i)
            data.append(1.0)
            continue

        # vzamemo uteži povezav
        utezi = [podatki["weight"] for _, _, podatki in izhodne]
        skupna_vsota = sum(utezi)

        # Normaliziramo uteži v verjetnosti (p_i = weight_i / sum_weights)
        for (_, v, _), w in zip(izhodne, utezi):
            j = node_to_index[v]
            row_ind.append(i)
            col_ind.append(j)
            data.append(w / skupna_vsota)

    P = sp.csr_matrix((data, (row_ind, col_ind)), shape=(n, n))
    return P, index_to_node, node_to_index

def vec_M_sparse_bicgstab(P):
    """
    Izračuna matriko M pričakovanih časov srečanja z metodo bicgstab
    """
    n = P.shape[0]
    n2 = n * n
    
    # 1. Kroneckerjev produkt P \otimes P
    P_kron = sp.kron(P, P, format='csr')
    
    # 2. Diagonalna matrika E = diag(1_{n^2} - vec(I_n))
    diag_E = np.ones(n2, dtype=float)
    srecanja_indices = [k * n + k for k in range(n)]
    diag_E[srecanja_indices] = 0.0
    E = sp.diags(diag_E, format='csr')
    
    # 3. Zmnožek (P \otimes P) * E
    PPE = P_kron.dot(E)
    
    # 4. Sistem A = I_{n^2} - (P \otimes P) * E
    I_n2 = sp.eye(n2, format='csr')
    A = I_n2 - PPE
    b = np.ones(n2, dtype=float)
    
    # 5. Reševanje sistema A * vec(M) = 1_{n^2}
    vec_M, info = bicgstab(A, b, rtol=1e-8, maxiter=2000)
    
    if info != 0:
        print(f"Opozorilo: bicgstab ni popolnoma konvergiral (koda: {info}).")
        
    M = vec_M.reshape((n, n))
    return M

### Izračun za model 3

In [24]:
GRAF_PATH = "model 3/model3_cakanje.graphml"
graf = nx.read_graphml(GRAF_PATH)

In [ ]:
# 1. KORAK: Gradnja stohastične matrike P
start_p = time.perf_counter()
P, index_to_node, node_to_index = zgradi_multistohasticno_matriko(graf)
cas_p = time.perf_counter() - start_p

print(f"--- 1. KORAK ZAKLJUČEN ---")
print(f"Čas gradnje matrike P: {cas_p:.4f} sekund")
print(f"Dimenzija matrike P: {P.shape[0]} x {P.shape[1]}\n")

# 2. KORAK: Izračun matrike pričakovanih časov srečanja M
start_m = time.perf_counter()
M3 = vec_M_sparse_bicgstab(P)
cas_m = time.perf_counter() - start_m

print(f"--- 2. KORAK ZAKLJUČEN ---")
print(f"Čas izračuna matrike M z bicgstab: {cas_m:.4f} sekund")


--- 1. KORAK ZAKLJUČEN ---
Čas gradnje matrike P: 0.0184 sekund
Dimenzija matrike P: 857 x 857

--- 2. KORAK ZAKLJUČEN ---
Čas izračuna matrike M z bicgstab: 12.6632 sekund


Pričakovani čas srečanja za začetni par vozlišč "Jadranska" in "Klinični center":

In [ ]:
j = node_to_index['603011']
k = node_to_index['402031']

pricakovani_cas_srecanja = M3[j, k]
pricakovani_cas_srecanja

np.float64(650.6464797088627)

### Izračun za model 2

In [41]:
GRAF_PATH = "model 2/model2_zdruzene_postaje.graphml"
graf = nx.read_graphml(GRAF_PATH)

In [42]:
### model 2 ###
# 1. KORAK: Gradnja stohastične matrike P
start_p = time.perf_counter()
P, index_to_node, node_to_index = zgradi_stohasticno_matriko(graf)
cas_p = time.perf_counter() - start_p

print(f"--- 1. KORAK ZAKLJUČEN ---")
print(f"Čas gradnje matrike P: {cas_p:.4f} sekund")
print(f"Dimenzija matrike P: {P.shape[0]} x {P.shape[1]}\n")

# 2. KORAK: Izračun matrike pričakovanih časov srečanja M
start_m = time.perf_counter()
M2 = vec_M_sparse_bicgstab(P)
cas_m = time.perf_counter() - start_m

print(f"--- 2. KORAK ZAKLJUČEN ---")
print(f"Čas izračuna matrike M z bicgstab: {cas_m:.4f} sekund")


--- 1. KORAK ZAKLJUČEN ---
Čas gradnje matrike P: 0.0047 sekund
Dimenzija matrike P: 461 x 461

--- 2. KORAK ZAKLJUČEN ---
Čas izračuna matrike M z bicgstab: 3.8900 sekund


Pričakovani čas srečanja za začetni par vozlišč "Jadranska" in "Klinični center":

In [43]:
j = node_to_index['mega_368']
k = node_to_index['mega_114']

pricakovani_cas_srecanja = M2[j, k]
pricakovani_cas_srecanja

np.float64(568.8977517310259)

### Izračun za model 1

In [44]:
GRAF_PATH = "model 1/model1_frekvenca.graphml"
graf = nx.read_graphml(GRAF_PATH)

In [45]:
## model 1
# 1. KORAK: Gradnja stohastične matrike P
start_p = time.perf_counter()
P, index_to_node, node_to_index = zgradi_stohasticno_matriko(graf)
cas_p = time.perf_counter() - start_p

print(f"--- 1. KORAK ZAKLJUČEN ---")
print(f"Čas gradnje matrike P: {cas_p:.4f} sekund")
print(f"Dimenzija matrike P: {P.shape[0]} x {P.shape[1]}\n")

# 2. KORAK: Izračun matrike pričakovanih časov srečanja M
start_m = time.perf_counter()
M1 = vec_M_sparse_bicgstab(P)
cas_m = time.perf_counter() - start_m

print(f"--- 2. KORAK ZAKLJUČEN ---")
print(f"Čas izračuna matrike M z bicgstab: {cas_m:.4f} sekund")

--- 1. KORAK ZAKLJUČEN ---
Čas gradnje matrike P: 0.0073 sekund
Dimenzija matrike P: 857 x 857

--- 2. KORAK ZAKLJUČEN ---
Čas izračuna matrike M z bicgstab: 6.7812 sekund


Pričakovani čas srečanja za začetni par vozlišč "Jadranska" in "Klinični center":

In [46]:
j = node_to_index['603011']
k = node_to_index['402031']

pricakovani_cas_srecanja = M1[j, k]
pricakovani_cas_srecanja

np.float64(1639.6915804321534)